# Student Roster — Dataset Documentation

**Dataset:** `students.csv`

**Purpose:** Establishes the master student roster for the synthetic Graduation Tracker student information system. Provides unique student identifiers, grade levels, enrollment statuses, and expected graduation cohorts. Serves as the primary reference dataset for linking student academic records, Keystone assessments, Act 158 pathways, community service activities, and interventions.

**Record level:** One row per unique student.

**Primary key:** `student_id`

**Fields:**

- `student_id`: Unique, sequential numeric student identifier beginning at 1145269.
- `grade_level`: Student's current grade level (9, 10, 11, or 12).
- `enrollment_status`: Student's enrollment classification (Active, Dropped, Withdrawn, or Transferred).
- `cohort`: Expected graduation year calculated from the student's grade level using 2027 as the reference graduation year for 12th-grade students.

**Business rules:**

- The dataset contains 5,000 synthetic student records.
- Each student receives a unique student identifier.
- Student IDs are generated sequentially, beginning at 1145269.
- Grade levels are randomly assigned from grades 9–12 with equal probability.
- Enrollment statuses follow these probability distributions:
  - Active: 75%
  - Dropped: 8%
  - Withdrawn: 7%
  - Transferred: 10%
- Each student receives exactly one enrollment status.
- Graduation cohorts are calculated using the following formula:

  `cohort = 2027 + (12 - grade_level)`

- Graduation cohorts are assigned according to grade level:
  - Grade 12: Cohort 2027.
  - Grade 11: Cohort 2028.
  - Grade 10: Cohort 2029.
  - Grade 9: Cohort 2030.
- All student IDs must be unique.
- All grade levels must fall between 9 and 12.
- All enrollment statuses must match the four predefined categories.
- Graduation cohorts must match the calculated value for each student's grade level.
- No required fields may contain missing values.
- Random generation uses a fixed seed of 42 to support reproducibility.

**Data validation:**

- Verify the dataset contains exactly 5,000 records.
- Verify the number of unique student IDs equals the total number of records.
- Check for missing values across all fields.
- Review the distribution of students by grade level.
- Confirm that all grade levels are valid.
- Confirm that all enrollment statuses belong to the predefined categories.
- Verify that graduation cohorts are correctly calculated from grade levels.

**Data source:** Synthetic student roster generated programmatically using Python, pandas, and NumPy. No actual student information is used.

**Generation method:** Student IDs are generated sequentially using Python list comprehension. Grade levels and enrollment statuses are assigned using NumPy random sampling. Graduation cohorts are calculated using pandas column operations. A fixed random seed ensures reproducible results.

**Output location:** `Graduation_Tracker/data/ground_truth/students.csv`

**Limitations:** Student identifiers, grade levels, enrollment statuses, and graduation cohorts are synthetically generated and do not represent actual student records. Grade levels are assigned independently with equal probability and do not reflect real-world enrollment distributions. Enrollment statuses are randomly assigned and are not based on attendance, academic performance, or withdrawal histories. Graduation cohorts are calculated solely from current grade level and assume standard grade progression without retention, acceleration, or changes to expected graduation dates. The dataset does not include student enrollment dates, withdrawal dates, demographic characteristics, or historical enrollment changes. Enrollment status is not currently used to determine whether a student should be included in graduation readiness calculations.

In [ ]:
# import libraries

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files
import shutil

In [ ]:
# connect Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# create project folders

project_folder = Path(
    '/content/drive/MyDrive/Graduation_Tracker'
)

folders = [
    'data/ground_truth',
    'data/raw_dirty',
    'data/cleaned',
    'data/audit',
    'notebooks',
    'sql',
    'docs',
    'dashboards'
]

for folder in folders:
  (project_folder / folder).mkdir(
      parents = True,
      exist_ok = True
  )

shutil.move('/content/drive/MyDrive/Colab Notebooks', '/content/drive/MyDrive/Graduation_Tracker/notebooks')

'/content/drive/MyDrive/Graduation_Tracker/notebooks/Colab Notebooks'

In [ ]:
# Enter number of student records to be created

num_students = 5000

In [ ]:
# Set random seed

rng = np.random.default_rng(42)

In [ ]:
# Generate student IDs

student_ids = [i + 1145268 for i in range(1, num_students + 1)]

print(student_ids[:10])

[1145269, 1145270, 1145271, 1145272, 1145273, 1145274, 1145275, 1145276, 1145277, 1145278]


In [ ]:
# Generate grade level list

grade_levels = rng.choice(
    [9, 10, 11, 12],
    size = num_students
)

print(grade_levels[:10])

[ 9  9 11 11 12 12 10 12 11 10]


In [ ]:
# Generate enrollment status list

enrollment_status = rng.choice(
    ['Active', 'Dropped', 'Withdrawn', 'Transferred'],
    size = num_students,
    p = [0.75, 0.08, 0.07, 0.10]
)

print(enrollment_status[:10])

['Active' 'Active' 'Active' 'Active' 'Active' 'Active' 'Active'
 'Transferred' 'Active' 'Active']


In [ ]:
# Create students DataFrame

students = pd.DataFrame({
    'student_id': student_ids,
    'grade_level': grade_levels,
    'enrollment_status': enrollment_status
})

display(students.head(10))

,student_id,grade_level,enrollment_status
0,1145269,9,Active
1,1145270,9,Active
2,1145271,11,Active
3,1145272,11,Active
4,1145273,12,Active
5,1145274,12,Active
6,1145275,10,Active
7,1145276,12,Transferred
8,1145277,11,Active
9,1145278,10,Active


In [ ]:
# Assign graduation cohort

year = 2027

students['cohort'] = (
    year + (12 - students['grade_level'])
)

display(students.head(10))

,student_id,grade_level,enrollment_status,cohort
0,1145269,9,Active,2030
1,1145270,9,Active,2030
2,1145271,11,Active,2028
3,1145272,11,Active,2028
4,1145273,12,Active,2027
5,1145274,12,Active,2027
6,1145275,10,Active,2029
7,1145276,12,Transferred,2027
8,1145277,11,Active,2028
9,1145278,10,Active,2029


In [ ]:
# Check that the data was created accurately

# Number of students
print(len(students))

# Number of unique student ids
print(students['student_id'].nunique())

# Missing data
print(students.isnull().sum())

# Grade level counts
students['grade_level'].value_counts().sort_index()

5000
5000
student_id           0
grade_level          0
enrollment_status    0
cohort               0
dtype: int64


,count
grade_level,
9,1268
10,1263
11,1280
12,1189


In [ ]:
# Save dataset to Google Drive

file_path = (
    project_folder
    / 'data'
    / 'ground_truth'
    / 'students.csv'
)

students.to_csv(
    file_path,
    index = False
)

In [ ]:
# Download CSV to local drive

files.download(str(file_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Run final validation

assert len(students) == num_students

assert students['student_id'].is_unique

assert students['grade_level'].isin(
    [9, 10, 11, 12]
).all()

assert students['enrollment_status'].isin(
    ['Active', 'Dropped', 'Withdrawn', 'Transferred']
).all()

assert (
    students['cohort'] == year + (12 - students['grade_level'])
).all()